# 행렬 곱셈 단계별 추적기 - 단일 원소 계산부터 행렬 곱까지

- Tutorial ID: `ull-2`
- Tutorial: 행렬 곱셈 단계별 추적기
- Section ID: `ull-2-1`
- Section: 단일 원소 계산부터 행렬 곱까지


In [ ]:
# ============================================================
# 코드 읽는 법 — 단일 원소 계산부터 행렬 곱까지
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 62)
print("행렬 곱셈 단계별 추적")
print("=" * 62)

In [ ]:
# 1. 단일 원소 계산 추적

In [ ]:
print("\n1. 단일 원소 C[0,0] 계산 추적")
print("-" * 50)

A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 10],
              [11, 12]])

print(f"A (2×3):\n{A}")
print(f"\nB (3×2):\n{B}")
print(f"\nC = A @ B (2×2):")

# C[0,0] 단계별
i, j = 0, 0
print(f"\n  C[{i},{j}] = ", end="")
accumulator = 0
steps = []
for k in range(A.shape[1]):
    product = A[i, k] * B[k, j]
    accumulator += product
    steps.append(f"A[{i},{k}]×B[{k},{j}] = {A[i,k]}×{B[k,j]} = {product}")

print(" + ".join(f"{A[i,k]}×{B[k,j]}" for k in range(A.shape[1])))
for step in steps:
    print(f"         {step}")
print(f"         = {accumulator}")

In [ ]:
# 2. 전체 행렬 곱 추적

In [ ]:
print("\n2. 전체 행렬 곱 추적")
print("-" * 50)

C = np.zeros((A.shape[0], B.shape[1]))
total_macs = 0

for i in range(A.shape[0]):
        for j in range(B.shape[1]):
        print(f"\n  C[{i},{j}]:", end=" ")
        acc = 0
        terms = []
                for k in range(A.shape[1]):
            product = A[i, k] * B[k, j]
            acc += product
            total_macs += 1
            terms.append(f"{A[i,k]}×{B[k,j]}")
        C[i, j] = acc
        print(f"{' + '.join(terms)} = {acc}")

print(f"\n  결과 C:\n{C.astype(int)}")
print(f"\n  검증: np.allclose = {np.allclose(C, A @ B)}")
print(f"  총 MAC 연산: {total_macs}")
print(f"  총 FLOPS: {total_macs * 2} (곱 + 덧셈)")

In [ ]:
# 3. 어텐션 점수 계산 추적

In [ ]:
print("\n\n3. 어텐션 점수 Q·K^T 계산 추적")
print("-" * 50)

d_head = 3
seq_len = 2

np.random.seed(42)
Q = np.round(np.random.randn(seq_len, d_head), 2)
K = np.round(np.random.randn(seq_len, d_head), 2)

print(f"Q (query):\n{Q}")
print(f"\nK (key):\n{K}")
print(f"\nscale = 1/sqrt(d_head) = 1/sqrt({d_head}) = {1/np.sqrt(d_head):.4f}")

scores = np.zeros((seq_len, seq_len))
for i in range(seq_len):
        for j in range(seq_len):
        dot = 0
        print(f"\n  score[q{i},k{j}] = Q[{i}]·K[{j}] / sqrt({d_head})")
        terms = []
                for k in range(d_head):
            prod = Q[i, k] * K[j, k]
            dot += prod
            terms.append(f"({Q[i,k]:+.2f})×({K[j,k]:+.2f})")
        scaled = dot / np.sqrt(d_head)
        scores[i, j] = scaled
        print(f"    = ({' + '.join(terms)}) / {np.sqrt(d_head):.4f}")
        print(f"    = {dot:.4f} / {np.sqrt(d_head):.4f}")
        print(f"    = {scaled:.4f}")

print(f"\n  어텐션 점수 행렬:\n{np.round(scores, 4)}")

# 소프트맥스
def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attn = softmax(scores)
print(f"\n  소프트맥스 후 (어텐션 가중치):\n{np.round(attn, 4)}")

In [ ]:
# 4. 연산량 분석

In [ ]:
print("\n\n4. 트랜스포머 연산량 분석")
print("-" * 50)

configs = [
    ("Tiny (교육용)", 4, 32, 2, 2),
    ("GPT-2 Small", 12, 768, 12, 1024),
    ("GPT-3 175B", 96, 12288, 96, 2048),
]

for name, n_layers, d_model, n_heads, seq_len in configs:
        d_head = d_model // n_heads
    
    # 한 레이어의 어텐션 FLOPS
    qkv_flops = 3 * 2 * seq_len * d_model * d_model  # Q, K, V 투영
    attn_flops = 2 * n_heads * seq_len * seq_len * d_head  # Q·K^T
    attn_v_flops = 2 * n_heads * seq_len * seq_len * d_head  # A·V
    out_flops = 2 * seq_len * d_model * d_model  # 출력 투영
    
    layer_attn = qkv_flops + attn_flops + attn_v_flops + out_flops
    total = layer_attn * n_layers
    
    print(f"\n  {name}:")
    print(f"    QKV 투영:    {qkv_flops:>15,} FLOPS")
    print(f"    Q·K^T:       {attn_flops:>15,} FLOPS")
    print(f"    Attn·V:      {attn_v_flops:>15,} FLOPS")
    print(f"    출력 투영:   {out_flops:>15,} FLOPS")
    print(f"    레이어 총:   {layer_attn:>15,} FLOPS")
    print(f"    전체 ({n_layers}L): {total:>15,} FLOPS ({total/1e12:.2f} TFLOPS)")